# GNN Training — Transportation Network

**Goal:** Learn node embeddings for transit stops / urban POIs so that the LLM receives structured spatial context instead of raw coordinate lists.

**Standalone module — no dependency on the Map Chat backend.**

---

## Pipeline

```
Raw data (GTFS / OpenStreetMap / Google Places)
        ↓
Graph construction  (nodes = stops/POIs, edges = routes/proximity)
        ↓
Feature matrix  (coords, type, degree, ...)
        ↓
GNN (GraphSAGE / GCN / GAT)
        ↓
Node embeddings  →  saved to embeddings.pt / embeddings.csv
        ↓
(optional) inject into LLM prompt as structured context
```

## 0. Imports & config

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GCNConv, GATConv
from torch_geometric.utils import from_networkx

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load raw data

Options:
- **GTFS** — Israel MoT feed (`stops.txt`, `stop_times.txt`, `routes.txt`)
- **Google Places** — export from `get_nearby_transit` / `get_nearby_places` calls
- **OpenStreetMap** — via `osmnx`

In [ ]:
# ── Option A: load from CSV export of transit stops ──────────────────────────
# stops_df = pd.read_csv('data/stops.csv')   # columns: stop_id, name, lat, lon, type

# ── Option B: load exam places as seed nodes ─────────────────────────────────
places_df = pd.read_csv('../exam/Places.csv')
places_df.columns = places_df.columns.str.strip()
places_df = places_df.rename(columns={'X': 'lat', 'Y': 'lon', 'PLACE': 'name', 'INDEX': 'node_id'})
print(places_df.head())
print(f'Loaded {len(places_df)} nodes')

## 2. Build graph

In [ ]:
from sklearn.metrics.pairwise import haversine_distances
from math import radians

PROXIMITY_KM = 3.0  # connect nodes within this radius

coords_rad = places_df[['lat', 'lon']].apply(lambda c: c.map(radians)).values
dist_matrix = haversine_distances(coords_rad) * 6371  # km

G = nx.Graph()
for _, row in places_df.iterrows():
    G.add_node(row['node_id'], name=row['name'], city=row['city'],
               lat=row['lat'], lon=row['lon'])

node_ids = places_df['node_id'].tolist()
for i in range(len(node_ids)):
    for j in range(i + 1, len(node_ids)):
        if dist_matrix[i, j] <= PROXIMITY_KM:
            G.add_edge(node_ids[i], node_ids[j], weight=1.0 / (dist_matrix[i, j] + 1e-6))

print(f'Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}')

In [ ]:
# Quick visualisation
pos = {row['node_id']: (row['lon'], row['lat']) for _, row in places_df.iterrows()}
plt.figure(figsize=(10, 8))
nx.draw(G, pos=pos, node_size=60, node_color='steelblue', edge_color='#ccc', with_labels=False)
plt.title('Transportation / POI proximity graph')
plt.axis('equal')
plt.tight_layout()
plt.show()

## 3. Build feature matrix

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Normalise coordinates
scaler = StandardScaler()
coord_features = scaler.fit_transform(places_df[['lat', 'lon']].values)

# City as one-hot  (extend with stop type, degree, etc. as needed)
city_enc = LabelEncoder()
city_ids = city_enc.fit_transform(places_df['city'])
city_onehot = np.eye(len(city_enc.classes_))[city_ids]

X = np.hstack([coord_features, city_onehot]).astype(np.float32)
print(f'Feature matrix: {X.shape}')  # (N, F)

## 4. Convert to PyG Data object

In [ ]:
# Map original node_ids to 0-based indices
id2idx = {nid: i for i, nid in enumerate(node_ids)}
edge_list = [(id2idx[u], id2idx[v]) for u, v in G.edges()]
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()  # (2, E)
# Add reverse edges (undirected)
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

data = Data(
    x=torch.tensor(X, dtype=torch.float),
    edge_index=edge_index,
).to(DEVICE)

print(data)

## 5. Define GNN model

In [ ]:
class TransportGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return x  # node embeddings


IN_DIM  = X.shape[1]
HIDDEN  = 64
EMBED   = 32

model = TransportGNN(IN_DIM, HIDDEN, EMBED).to(DEVICE)
print(model)

## 6. Unsupervised training — graph autoencoder / link prediction loss

No labels needed: train with a **negative-sampling link prediction** objective.
Nodes that are connected should have similar embeddings.

In [ ]:
from torch_geometric.utils import negative_sampling

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 200

def link_pred_loss(z, pos_edge_index, neg_edge_index):
    pos_score = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=1)
    neg_score = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)
    labels = torch.cat([torch.ones(pos_score.size(0)), torch.zeros(neg_score.size(0))]).to(DEVICE)
    return F.binary_cross_entropy_with_logits(torch.cat([pos_score, neg_score]), labels)

losses = []
model.train()
for epoch in range(1, EPOCHS + 1):
    optimizer.zero_grad()
    z = model(data.x, data.edge_index)
    neg_edge = negative_sampling(data.edge_index, num_nodes=data.num_nodes,
                                 num_neg_samples=data.edge_index.size(1) // 2)
    loss = link_pred_loss(z, data.edge_index, neg_edge)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 50 == 0:
        print(f'Epoch {epoch:3d}  loss={loss.item():.4f}')

plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training loss')
plt.tight_layout(); plt.show()

## 7. Extract & save embeddings

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = model(data.x, data.edge_index).cpu().numpy()

print(f'Embeddings shape: {embeddings.shape}')  # (N, EMBED)

# Save as CSV for inspection / LLM injection
emb_df = places_df[['node_id', 'name', 'city', 'lat', 'lon']].copy()
for i in range(EMBED):
    emb_df[f'e{i}'] = embeddings[:, i]
emb_df.to_csv('embeddings.csv', index=False, encoding='utf-8-sig')
print('Saved embeddings.csv')

# Save raw tensor
torch.save(embeddings, 'embeddings.pt')
print('Saved embeddings.pt')

## 8. Visualise embeddings (UMAP / t-SNE)

In [ ]:
# pip install umap-learn
import umap

reducer = umap.UMAP(n_components=2, random_state=42)
proj = reducer.fit_transform(embeddings)

cities = places_df['city'].tolist()
unique_cities = list(set(cities))
color_map = {c: i for i, c in enumerate(unique_cities)}
colors = [color_map[c] for c in cities]

plt.figure(figsize=(9, 7))
scatter = plt.scatter(proj[:, 0], proj[:, 1], c=colors, cmap='tab10', s=80)
for i, row in places_df.iterrows():
    plt.annotate(row['name'][:10], (proj[i, 0], proj[i, 1]), fontsize=7, ha='center')
plt.title('Node embeddings — UMAP (colored by city)')
plt.tight_layout()
plt.show()

---
## Notes / Next steps

- [ ] Replace proximity graph with real GTFS route graph (shared route = edge)
- [ ] Add richer node features: stop type, route count, ridership, OSM tags
- [ ] Try GAT to learn attention weights over neighbour stops
- [ ] Evaluate embeddings: nearest-neighbour retrieval, city clustering purity
- [ ] Inject top-k nearest embedding neighbours into LLM prompt as structured context